# Imports de librerias

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import StringType
from pyspark.sql.functions import trim, col, lower, count, countDistinct, when, length, lpad, translate

# Lectura de la Tabla Bronce olist_customers

In [0]:
# creo el df con la tabla bronce de olist_customers
df = spark.table("`catalog_brazilian-e-commerce`.bronze.olist_customers")

# Transformaciones

In [0]:
# hay que explorar los datos para encontrar "errores" y ver que calidad tiene
df.display()

In [0]:
# filas del dataframe
df.count()

In [0]:
# filas de customer_id para ver la unicidad y que ningun id de compra se duplique
df.select("customer_id").distinct().count()
# comparando con el resultado de arriba da igual por lo que los id son todos distintos.

In [0]:
# se verifica si hay recurrencia de compras de los clientes
df.groupBy("customer_unique_id").count().orderBy("count", ascending=False).show()

In [0]:
# limpieza de espacios en extremos de customer_city y se pasan los textos a minuscula
df = df.withColumn(
    "customer_city",
    lower(trim("customer_city"))
)

In [0]:
# limpieza de espacios en extremos de customer_state
df = df.withColumn(
    "customer_state",
    trim("customer_state")
)

In [0]:
#  veo si hay espacios multiples
df.filter(col("customer_city").rlike(r"\s{2,}")).show(20, False)

In [0]:
# veo si hay espacios al inicio y al final
df.filter(col("customer_city") != trim(col("customer_city"))).show(20, False)

In [0]:
# veo si hay mayusculas mezcladas
df.filter(col("customer_city") != lower(col("customer_city"))).show(20, False)

In [0]:
# detectar valores negativos o longitudes raras (ver si impacta en algo que el min tenga 4 digitos en vez de 5)
df.select("customer_zip_code_prefix").describe().show()

In [0]:
# deteccion de valores nulls
df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

In [0]:
# cardinalidad de customer_city
df.select("customer_city").distinct().count()


In [0]:
# cardinalidad de customer_state
df.select("customer_state").distinct().count()

In [0]:
# valido que no haya errores en customer_state y coinciden con los 27 estados de brasil
df.select("customer_state").distinct().show(30)

In [0]:
# normaliza customer_city ante errores en acentos
df = df.withColumn(
    "customer_city",
    translate(col("customer_city"),
              "áàãâäéèêëíìîïóòõôöúùûüç",
              "aaaaaeeeeiiiiooooouuuuc")
)

In [0]:
# el zip_code_prefix debe tener 5 digitos
df = df.withColumn(
    "customer_zip_code_prefix",
    lpad(col("customer_zip_code_prefix").cast("string"), 5, "0")
)

# Crear la tabla Silver de olist_customers

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("`catalog_brazilian-e-commerce`.silver.olist_customers")

In [0]:
%sql
select * from `catalog_brazilian-e-commerce`.silver.olist_customers